# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIRˆ²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) mass spectrometry dataset using the `mlcroissant` library in Python.

### Dataset Source
The dataset is described by a Croissant schema available at a public URL. The Croissant metadata describes all record sets, fields, and relevant entities using `@id` identifiers for precise referencing.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset's metadata and examine its high-level properties using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load Croissant Dataset object
dataset = mlc.Dataset(croissant_url)

# Access metadata
metadata = dataset.metadata
print(f"Dataset name: {getattr(metadata, 'name', None)}\n")
print(f"Description: {getattr(metadata, 'description', None)}\n")
print(f"Published: {getattr(metadata, 'datePublished', None)}")

## 2. Data Overview
Review available record sets, their fields, columns, and associated `@id`s for precise referencing. This overview helps in selecting the relevant data for analysis.

In [ ]:
# List all record sets and their details using @id

# Extract all record sets from the metadata
record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    print("No record sets defined in the metadata, looking for datasets under distribution...")
    # Try to find tabular data under distribution
    distribution = getattr(metadata, 'distribution', [])
    files_with_columns = []
    for dist in distribution:
        dist_obj = dist
        columns = getattr(dist_obj, 'column', []) if hasattr(dist_obj, 'column') else []
        if columns:
            files_with_columns.append((getattr(dist_obj, '@id', None), columns))
    if files_with_columns:
        print("Record set(s) as data file objects (using distributions with columns):\n")
        for file_id, columns in files_with_columns:
            print(f"Data file object @id: {file_id}")
            for col in columns:
                col_id = getattr(col, '@id', None)
                col_name = getattr(col, 'name', None)
                col_type = getattr(col, 'dataType', None)
                print(f"  Column @id: {col_id} | name: {col_name} | dataType: {col_type}")
        print("\nNote: Croissant package encodes tables as DataFileObjects when explicit recordSet absent.")
    else:
        print("No record sets or columns detected in the metadata. Please check dataset.")
else:
    print("Record sets detected:\n")
    for rs in record_sets:
        rs_id = getattr(rs, '@id', None)
        rs_name = getattr(rs, 'name', None)
        print(f"  RecordSet @id: {rs_id} | name: {rs_name}")
        fields = getattr(rs, 'field', [])
        for fld in fields:
            fld_id = getattr(fld, '@id', None)
            fld_name = getattr(fld, 'name', None)
            fld_type = getattr(fld, 'dataType', None)
            print(f"    Field @id: {fld_id} | name: {fld_name} | dataType: {fld_type}")

## 3. Data Extraction
Load records from a selected record set (or data file object via `@id`) into a pandas DataFrame for further inspection. Use the `@id` found above to select your source.

Below, we find the main table identifier and load the table.

In [ ]:
# Identify the correct data file @id (from overview)
# Here, we'll dynamically pick the first data file object with columns as the main record set
distribution = getattr(metadata, 'distribution', [])
main_record_set_id = None
main_columns = []
for dist in distribution:
    columns = getattr(dist, 'column', []) if hasattr(dist, 'column') else []
    if columns:
        main_record_set_id = getattr(dist, '@id', None)
        main_columns = columns
        break

if not main_record_set_id:
    raise ValueError("No tabular data found in dataset distribution.")

print(f"Main DataFileObject @id: {main_record_set_id}")
print("Columns available in the data file:\n")
for col in main_columns:
    print(f"  Column @id: {getattr(col, '@id', None)} | name: {getattr(col, 'name', None)} | dataType: {getattr(col, 'dataType', None)}")

# Now, load the table as a DataFrame
df = pd.DataFrame(list(dataset.records(record_set=main_record_set_id)))
print(f"\nColumns in dataframe:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
We demonstrate typical EDA: filtering by a numeric field, normalization, and grouping by categorical fields.

First, identify a numeric field (`@id`) and a grouping field from the columns above.

In [ ]:
# Choose one numeric field and one group field from the listed columns or metadata.
# Example: Suppose column names include 'MW [Da]' (molecular weight, numeric) and 'SampleGroup' (categorical).
# We refer to their @id when possible, but since column names are as in the data, use those.

# Find a numeric field
import numpy as np

# Try to pick a numeric column (e.g. 'MW [Da]')
possible_numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if not possible_numeric_cols:
    # Try to parse columns containing digits
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
            possible_numeric_cols.append(col)
        except Exception:
            pass
    possible_numeric_cols = list(set(possible_numeric_cols))

if not possible_numeric_cols:
    raise ValueError("No numeric fields found in main table; cannot proceed with numeric analysis.")

numeric_field = possible_numeric_cols[0]

print(f"Numeric field (@id or name): {numeric_field}")

# Pick a group/categorical field
group_field = None
for col in df.columns:
    if col.lower().startswith('sample') or col.lower().startswith('group') or len(df[col].unique()) < 10 and df[col].dtype == object:
        group_field = col
        break
if not group_field:
    group_field = df.columns[0]  # Use first column as fallback
print(f"Group field (@id or name): {group_field}")

# Filter: pick threshold for the numeric field
threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 10
filtered_df = df[df[numeric_field] > threshold].copy()

print(f"Filtered records with {numeric_field} > {threshold:.2f} (total: {len(filtered_df)}):")
display(filtered_df.head())

# Normalization (z-score)
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by group_field, show means
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(f"mean_{numeric_field}")
    print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
    display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field, and compare by group where possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Boxplot/grouped by category
if group_field and group_field in df.columns and df[group_field].nunique() < 20:
    plt.figure(figsize=(10,6))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
- Successfully loaded and explored the [FAIRˆ² Mass Spectrometry Dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa).
- Identified key fields and demonstrated how to reference all record sets, columns, and fields using their `@id`.
- Loaded tabular data via its `@id` using `mlcroissant`, and performed basic filtering, normalization, and visualization of protein-level properties.
- The Croissant schema enables robust, discoverable data exploration workflows referencing precise schema elements.

You can now use this notebook as a template for further analysis, or for programmatic extraction of specific fields as referenced by their unique `@id`.